# AgrIA · Vérification des données sur Colab

Étape 1 de `docs/ARCHITECTURE.md`. Ce notebook vérifie que `agria_processed.zip` s'ouvre sur Colab et que toutes les images des manifests sont lisibles sous Linux.

**Avant de lancer :** menu *Exécution > Modifier le type d'exécution > GPU*, puis indique ci-dessous où se trouve l'archive sur ton Drive.

In [ ]:
ZIP_PATH = "/content/drive/MyDrive/AgrIA/agria_processed.zip"  # à adapter
ROOT = "/content/AgrIA"

## 1. GPU et Drive

In [ ]:
import torch
from google.colab import drive

print("GPU :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "AUCUN : active le GPU")
drive.mount("/content/drive")

## 2. Décompression

On décompresse sur le disque de Colab, pas sur le Drive : la lecture des images y est bien plus rapide. Ce disque est effacé à la fin de la session, il faudra relancer cette cellule.

In [ ]:
import os
import shutil
import zipfile

assert os.path.exists(ZIP_PATH), f"Archive introuvable : {ZIP_PATH}"
print(f"Archive : {os.path.getsize(ZIP_PATH) / 1e9:.1f} Go, disque libre : {shutil.disk_usage('/content').free / 1e9:.1f} Go")
if not os.path.exists(os.path.join(ROOT, "data", "processed", "routing_manifest.csv")):
    os.makedirs(ROOT, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(ROOT)
print(os.listdir(os.path.join(ROOT, "data", "processed")))

## 3. Contrôle des manifests

Pour chaque fichier : les chemins sont convertis de `\` à `/`, toutes les images doivent exister, et un échantillon de 200 est réellement ouvert. Le résultat attendu est `missing 0` partout, avec des images en `(224, 224)`, `RGB`.

In [ ]:
import os
import random
from collections import Counter

import pandas as pd
from PIL import Image

PROC = os.path.join(ROOT, "data", "processed")
FILES = ["routing_manifest.csv", "diseases_manifest.csv", "deficiency_manifest.csv", "fruits_manifest.csv",
         "balanced/diseases_train.csv", "balanced/deficiency_train.csv", "balanced/fruits_train.csv"]


def load(name):
    """A manifest with its Windows paths turned into absolute paths of this machine."""
    df = pd.read_csv(os.path.join(PROC, name))
    df["path"] = [os.path.join(ROOT, *p.replace("\\", "/").split("/")) for p in df["path"]]
    return df


problems = 0
for name in FILES:
    df = load(name)
    missing = df[~df["path"].map(os.path.exists)]
    sample = df.sample(min(200, len(df)), random_state=42)
    sizes, bad = Counter(), 0
    for path in sample["path"]:
        try:
            with Image.open(path) as im:
                im.load()
                sizes[(im.size, im.mode)] += 1
        except Exception:
            bad += 1
    problems += len(missing) + bad
    print(f"{name:32} {len(df):>6} rows | missing {len(missing):>5} | unreadable in sample {bad:>3} | {dict(sizes)}")
    print("   ", df["split"].value_counts().to_dict() if "split" in df else df["origin"].value_counts().to_dict())
    if len(missing):
        print("    example:", missing["path"].iloc[0])
print("\nOK: every image is there and readable" if not problems else f"\nPROBLEM: {problems} files to look at")


## 4. Contrôle à l'œil

Chaque ligne doit correspondre à sa classe : des feuilles, des fruits, puis des photos sans plante.

In [ ]:
import matplotlib.pyplot as plt

routing = load("routing_manifest.csv")
classes = ["leaf", "fruit", "other"]
fig, axes = plt.subplots(3, 6, figsize=(14, 7))
for row, cls in zip(axes, classes):
    for ax, path in zip(row, routing[routing["class"] == cls].sample(6, random_state=1)["path"]):
        ax.imshow(Image.open(path))
        ax.set_title(cls)
        ax.axis("off")
plt.tight_layout()
plt.show()